# SI robustness: exploring the shipped results

This notebook reads only shipped repository files under `data/derived/`. It retrains nothing and makes no network calls. The asserted manuscript numbers live in `tests/test_sibench_parity.py`; this notebook is for looking around.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while not (REPO / "data" / "derived").exists() and REPO != REPO.parent:
    REPO = REPO.parent
DV = REPO / "data" / "derived"
sorted(p.name for p in DV.iterdir())

## Mechanism-channel sensitivity

The combined graph merges a broad evidence channel (GO, HPO, HPA, Pfam) with a specific mechanism-module channel. The counts below come from the built graph.

In [ ]:
st = json.loads((DV / "mechanism_channels" / "channel_stats.json").read_text())
pd.Series({k: st[k] for k in ["broad_nodes", "broad_edges", "spec_nodes", "spec_edges", "overlap_edges", "union_nodes", "union_edges"]})

In [ ]:
pd.read_csv(DV / "mechanism_channels" / "broad_edges_by_class.tsv", sep="\t", index_col=0)

## 31-arm ablation

Each arm retrains five seeds with one component removed or changed. The columns give the paired change in test-split AUROC against the released model, with a disease-cluster bootstrap CI (B = 2000).

In [ ]:
abl = pd.read_csv(DV / "ablation" / "ablation_paired_delta_vs_released_B2000.tsv", sep="\t").query("case_set == 'test'")
abl = abl.sort_values("d_auroc")[["arm", "d_auroc", "d_auroc_ci_lo", "d_auroc_ci_hi", "ci_excludes_0"]]
fig, ax = plt.subplots(figsize=(6, 7))
ax.errorbar(abl.d_auroc, range(len(abl)), xerr=[abl.d_auroc - abl.d_auroc_ci_lo, abl.d_auroc_ci_hi - abl.d_auroc], fmt="o", ms=3)
ax.set_yticks(range(len(abl)), abl.arm, fontsize=7); ax.axvline(0, color="grey", ls="--", lw=0.6)
ax.set_xlabel("change in test-split AUROC (95% CI)"); fig.tight_layout()
abl.head(10)

In [ ]:
knn = pd.read_csv(DV / "ablation" / "knn50_overlap_summary.tsv", sep="\t")
knn.head(10)

## Threshold sensitivity

The graph was rebuilt at several edge cut-offs and retrained. These are the shipped summary and per-seed results.

In [ ]:
pd.read_csv(DV / "threshold_sensitivity" / "threshold_sweep_values.tsv", sep="\t")

In [ ]:
pd.read_csv(DV / "threshold_sensitivity" / "threshold_sweep_per_seed.tsv", sep="\t").pivot_table(index="cutoff", values="auroc_all", aggfunc=["mean", "std"])

## Node-disjoint subsets and negative-set realism

The IxIDN-Matched pairs are scored per pair. The subgroup estimates share one bootstrap draw (B = 2000). For realism, we compare the ontology distance of benchmark negatives with that of random disease pairs.

In [ ]:
pp = pd.read_csv(DV / "ixidn_robustness" / "ixidn4364_per_pair_scores.tsv.gz", sep="\t")
pp.groupby("y")["p"].describe()

In [ ]:
pd.read_csv(DV / "ixidn_robustness" / "subgroups_shared_bootstrap_B2000.tsv", sep="\t")

In [ ]:
onto = pd.read_csv(DV / "ixidn_robustness" / "ixidn4364_ontology_distance.tsv", sep="\t").merge(pp[["blind_id", "y"]], on="blind_id")
rnd = pd.read_csv(DV / "ixidn_robustness" / "random_pairs_ordo_distance.tsv.gz", sep="\t")
fig, ax = plt.subplots(figsize=(5, 3))
for lab, s in [("IxIDN+", onto.query("y == 1").ordo_dist), ("IxIDN-", onto.query("y == 0").ordo_dist)] + [(k, g.ordo_dist) for k, g in rnd.groupby("set")]:
    ax.hist(s.dropna(), bins=range(0, 16), histtype="step", density=True, label=lab)
ax.set_xlabel("ORDO distance"); ax.legend(fontsize=7); fig.tight_layout()

## Uncertainty

This section compares seed-to-seed variation with pair-sampling variation, and pair-level with disease-cluster bootstrap CIs.

In [ ]:
pd.read_csv(DV / "uncertainty" / "seed_vs_sample_sd.tsv", sep="\t").query("metric == 'auroc'")[["stratum", "ensemble", "seed_sd", "sample_sd_pair", "sample_sd_cluster"]]

In [ ]:
pd.read_csv(DV / "uncertainty" / "ci_pair_vs_cluster.tsv", sep="\t").query("metric == 'auroc'")